# Examen ICCD753 Recuperación de Información

### Estudiante:
- Jose Luis Andino

### Fecha de Entrega:
- miércoles, 18 de noviembre de 2026, 13:15


## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado

Cada estudiante recibirá un conjunto de datos. Todos contienen documentos, consultas y qrels. Se utilizarán los identificadores originales de los documentos para que los resultados puedan evaluarse.

| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |
| SciFact | [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact) | [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels) |

## 3. Requisitos del sistema

### 3.1. Preparación e indexación

El sistema deberá:

1. Cargar e indexar los documentos del corpus asignado.
2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.
3. Construir un vocabulario y un índice invertido que permita recuperar documentos.
4. Conservar la relación entre cada documento indexado y su identificador original.

La unidad de evaluación será el documento original, porque los qrels relacionan consultas con identificadores de documentos. No se debe dividir el texto en fragmentos.

### 3.2. Modelos de recuperación

Para cualquier consulta de texto, el sistema deberá producir dos rankings independientes:

- TF-IDF con similitud coseno.
- BM25.

Cada ranking deberá ordenar los documentos por puntuación de relevancia y permitir solicitar los primeros `k` resultados.

### 3.3. Servicio web

El sistema deberá estar publicado en internet y ofrecer, como mínimo, los siguientes endpoints:

**Comprobación de disponibilidad**

```http
GET /health
```

**Búsqueda**

```http
POST /search
Content-Type: application/json
```

Ejemplo de solicitud:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` deberá aceptar `bm25` y `tfidf`. Una respuesta tendrá esta estructura:

```json
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "identificador_original",
      "score": 8.42,
      "title": "Título, si está disponible",
      "snippet": "Fragmento del documento recuperado"
    }
  ]
}
```

Los resultados deberán estar ordenados de mayor a menor puntuación, no contener documentos duplicados y respetar el valor solicitado de `top_k`. El servicio debe aceptar consultas que el estudiante no conozca de antemano.

## 4. Evaluación de la recuperación

Cada estudiante debe evaluar ambos modelos con consultas de la partición `test` y sus qrels. Presentará, como mínimo:

- Precision@10.
- Recall@10.
- NDCG@10.
- Una comparación de los resultados de TF-IDF y BM25.

En el Jupyter Notebook se deberá indicar qué consultas se utilizaron y permitir reproducir los cálculos. La evaluación del docente podrá utilizar otras consultas del mismo conjunto.

## 5. Prueba del docente

El docente accederá a la URL entregada y realizará solicitudes desde un cliente de web service para comprobar:

1. La disponibilidad del servicio.
2. El funcionamiento de TF-IDF y BM25.
3. El formato, orden y validez de los resultados.
4. La correspondencia de los `document_id` con el corpus y los qrels.
5. La respuesta ante consultas no incluidas en los ejemplos de entrega.

## 6. Entregables

Cada estudiante entregará:

1. La URL pública del servicio web, accesible durante el período de revisión.
2. El código en Jupyter Notebook con los resultados de la ejecución de cada bloque de código, incluyendo los resultados de las métricas de evaluación de los modelos de recuperación.

Se evaluará que ambos modelos funcionen correctamente y que cada estudiante interprete sus resultados.

## Corpus asignado: SciFact

SciFact contiene documentos científicos con su identificador original, consultas y juicios de relevancia. Se usa la partición `test` indicada por la consigna. Se indexan los artículos completos; no se generan fragmentos.

### Preprocesamiento

Los títulos y textos se concatenan para indexar cada documento. El analizador convierte a minúsculas, extrae tokens alfabéticos, quita stopwords inglesas y aplica stemming Porter. TF-IDF usa pesos normalizados y coseno; BM25 usa `k1=1.5`, `b=0.75` y la frecuencia documental inversa Robertson–Sparck Jones.


Antes de iniciar, instala las dependencias en el entorno del notebook desde una terminal con `uv sync` (o `pip install -r requirements.txt`). Después ejecuta las celdas en orden.


In [1]:
from examen01_jla.data import load_scifact
from examen01_jla.retrieval import SearchIndex, evaluate

documents, queries, qrels = load_scifact("test")
index = SearchIndex(documents)
print(f"Documentos: {len(documents):,} | vocabulario: {index.vocabulary_size:,}")
print(f"Consultas test: {len(queries):,} | consultas con qrels: {len(qrels):,}")


[INFO] Opening /home/andro/.ir_datasets/beir/scifact/docs.pklz4/bin with direct file access


Documentos: 5,183 | vocabulario: 21,099
Consultas test: 300 | consultas con qrels: 300


## Búsqueda de ejemplo

La consulta elegida se toma del conjunto de prueba para que el ejemplo sea reproducible. La evaluación usa todas las consultas con qrels más adelante.


In [2]:
query_id = next(iter(queries))
query = queries[query_id]
print("Consulta:", query_id, query)

for model in ("tfidf", "bm25"):
    print(f"\n{model.upper()}")
    for result in index.search(query, model=model, top_k=5):
        print(result)


Consulta: 1 0-dimensional biomaterials show inductive properties.

TFIDF
{'rank': 1, 'document_id': '10608397', 'score': 0.10159813632451727, 'title': 'High-performance neuroprosthetic control by an individual with tetraplegia.', 'snippet': 'BACKGROUND Paralysis or amputation of an arm results in the loss of the ability to orient the hand and grasp, manipulate, and carry objects, functions that are essential for activities of daily living. Brain-machine interfaces could provid…'}
{'rank': 2, 'document_id': '31543713', 'score': 0.09598119298501197, 'title': 'Data analysis methods for detection of differential protein expression in two-dimensional gel electrophoresis.', 'snippet': 'The recent development of microarray technology has led statisticians and bioinformaticians to develop new statistical methodologies for comparing different biological samples. The objective is to identify a small number of differentially…'}
{'rank': 3, 'document_id': '42421723', 'score': 0.09441686665326611, 

## Evaluación sobre qrels test

Precision@10 y Recall@10 consideran relevante todo documento con grado mayor que cero. NDCG@10 conserva los grados de relevancia y usa ganancia `2^relevancia - 1`. Se reporta el promedio macro entre las consultas evaluables.


In [3]:
metrics = evaluate(index, queries, qrels, k=10)
metrics


{'tfidf': {'precision@10': 0.08333333333333333,
  'recall@10': 0.7617222222222223,
  'ndcg@10': 0.606410933862849},
 'bm25': {'precision@10': 0.092,
  'recall@10': 0.8336666666666667,
  'ndcg@10': 0.6956382045815169}}

## Interpretación

| Modelo | Precision@10 | Recall@10 | NDCG@10 |
| --- | --- | --- | --- |
| TF-IDF (coseno) | 0.0833 | 0.7617 | 0.6064 |
| BM25 | 0.0920 | 0.8337 | 0.6956 |

BM25 supera a TF-IDF en las tres métricas. La mejora más clara está en NDCG@10 (+0.089, ≈15 % relativo) y en Recall@10 (+0.072): BM25 encuentra más documentos relevantes y además los coloca en posiciones más altas. Esto es coherente con sus ventajas teóricas: la saturación de la frecuencia del término (`k1`) evita que un término repetido domine la puntuación, y la normalización por longitud (`b`) trata mejor la variación de tamaño entre resúmenes científicos, mientras que TF-IDF con coseno no satura la frecuencia del término y su normalización por la norma del vector no es ajustable.

Precision@10 es baja en ambos modelos por la naturaleza de SciFact y no por un fallo del sistema: cada consulta (una afirmación científica) tiene alrededor de un documento relevante en los qrels, por lo que incluso un ranking perfecto tendría Precision@10 cercana a 0.1. Por eso el Recall@10 y el NDCG@10 son las métricas más informativas aquí: con BM25 el documento relevante aparece entre los diez primeros en cerca del 83 % de los casos.

Limitaciones: el preprocesamiento descarta los dígitos (por ejemplo `p53`, `IL-6`) y no maneja sinónimos ni paráfrasis, que es donde los modelos léxicos fallan frente a modelos semánticos.

## API

La aplicación definida en `examen01_jla/api.py` publica `GET /health` y `POST /search`. Ejemplo de cuerpo: `{"query":"...","model":"bm25","top_k":10}`. Para iniciarla localmente: `uvicorn examen01_jla.api:app --host 0.0.0.0 --port 8000`. La URL pública debe añadirse al entregar una vez desplegado el servicio.
